# 07 · Final model and the one-time test evaluation (plan steps 4.3–4.4, D-015)

**Run this once, after all training is finished.** Nothing is trained, tuned or selected after it.

1. **Selection (4.3)** among the MobileNetV2 runs on **validation** macro-F1 (ties → smaller model).
2. **Test (4.4)**: the selected model and the model already chosen in every other family (logistic regression,
   linear SVM, CNN from scratch) are scored on the test split, each on
   * the **full test split** (10,709 images, 38 classes) and
   * its **leaf-verified part** (8,033 images, 30 classes), where same-leaf leakage is impossible by construction.
3. CPU time per image (batch 1) and the real model size are measured for every model.

`plantdx.evaluate` refuses to score a run on test a second time. A **T4 GPU** runtime makes prediction faster;
test features for the classical models take a few minutes on the CPU.

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # the Colab copy is disposable: always match GitHub exactly (D-009)
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
head = !git -C {REPO_DIR} rev-parse HEAD
remote = !git -C {REPO_DIR} rev-parse origin/main
assert head == remote, "repo is not at the latest commit: run this cell again"
!git -C {REPO_DIR} log -1 --oneline
!nvidia-smi -L || echo "no GPU"
!python -m plantdx.env

## 2 · Images

In [ ]:
from plantdx.data.images import download_archive, extract_variant

VARIANT, DATA_ROOT = "color", "/content/data"
print("new files extracted:", extract_variant(download_archive(), DATA_ROOT, VARIANT))

## 3 · Select and evaluate

In [ ]:
from plantdx.evaluate import select_and_evaluate

FINAL_CANDIDATES = ["mobilenetv2_head", "mobilenetv2_ft", "mobilenetv2_ft_lr3e-5"]
OTHER_FAMILIES = ["baseline_logreg", "baseline_linsvm", "cnn_scratch"]

FINAL, table = select_and_evaluate(DATA_ROOT, FINAL_CANDIDATES, OTHER_FAMILIES)

## 4 · Comparison table

In [ ]:
import pandas as pd

cols = ["macro_f1", "accuracy", "top3"]
wide = table.pivot_table(index="run_id", columns="split", values=cols)
wide.columns = [f"{split}_{metric}" for metric, split in wide.columns]
order = ["val_macro_f1", "test_macro_f1", "test_verified_macro_f1", "val_accuracy", "test_accuracy",
         "test_verified_accuracy", "test_top3"]
info = table[table.split == "val"].set_index("run_id")[["params", "size_mb", "cpu_ms"]]
summary = wide[[c for c in order if c in wide]].join(info).sort_values("val_macro_f1", ascending=False)
summary.round(4)

## 5 · Download

`final_evaluation.zip` → send it in the chat. `inference.keras` of the final model is downloaded separately:
keep it for the GitHub Release (v1.0).

In [ ]:
from google.colab import files
from plantdx.paths import run_dir
from plantdx.results import export_for_repo, results_path

runs = [FINAL, *OTHER_FAMILIES]
out = export_for_repo(runs, "/content/final_evaluation.zip",
                      extra_reports=[results_path().parent / "final_model.json"])
files.download(str(out))
files.download(str(run_dir(FINAL) / "inference.keras"))